# Sample Databricks Notebook

This notebook demonstrates a simple data processing workflow that can be embedded in Databricks jobs.


In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
from datetime import datetime

print(f"Notebook started at: {datetime.now()}")

In [ ]:
# Configuration parameters (these can be passed from Databricks job)
# Databricks widgets for parameter passing
try:
    # These will work in Databricks environment
    dbutils.widgets.text("input_path", "/tmp/input", "Input Path")
    dbutils.widgets.text("output_path", "/tmp/output", "Output Path")
    dbutils.widgets.text("environment", "dev", "Environment")
    
    input_path = dbutils.widgets.get("input_path")
    output_path = dbutils.widgets.get("output_path")
    environment = dbutils.widgets.get("environment")
except:
    # Fallback for local testing
    input_path = "/tmp/input"
    output_path = "/tmp/output"
    environment = "local"

print(f"Configuration:")
print(f"  Input Path: {input_path}")
print(f"  Output Path: {output_path}")
print(f"  Environment: {environment}")

In [ ]:
# Sample data processing function
def process_data(data):
    """
    Sample data processing function
    """
    # Add timestamp
    data['processed_at'] = datetime.now()
    
    # Add some calculated fields
    if 'value' in data.columns:
        data['value_squared'] = data['value'] ** 2
        data['value_log'] = np.log1p(data['value'].abs())
    
    return data

# Create sample data for demonstration
sample_data = pd.DataFrame({
    'id': range(1, 101),
    'value': np.random.randn(100),
    'category': np.random.choice(['A', 'B', 'C'], 100)
})

print(f"Sample data shape: {sample_data.shape}")
print(sample_data.head())

In [ ]:
# Process the data
processed_data = process_data(sample_data.copy())

print(f"Processed data shape: {processed_data.shape}")
print(processed_data.head())

In [ ]:
# Summary statistics
summary = {
    'total_records': len(processed_data),
    'categories': processed_data['category'].nunique(),
    'avg_value': processed_data['value'].mean(),
    'processing_time': datetime.now()
}

print("Processing Summary:")
for key, value in summary.items():
    print(f"  {key}: {value}")

In [ ]:
# Save results (in Databricks, this would save to DBFS or cloud storage)
try:
    # In Databricks environment
    processed_data.to_csv(f"{output_path}/processed_data.csv", index=False)
    
    # Also save as Spark DataFrame in Databricks
    spark_df = spark.createDataFrame(processed_data)
    spark_df.write.mode("overwrite").parquet(f"{output_path}/processed_data.parquet")
    
    print(f"Data saved to {output_path}")
except:
    # Local testing - save to current directory
    processed_data.to_csv("processed_data_local.csv", index=False)
    print("Data saved locally as processed_data_local.csv")

In [ ]:
# Final status
print(f"\n{'='*50}")
print(f"Notebook execution completed successfully!")
print(f"Environment: {environment}")
print(f"Records processed: {len(processed_data)}")
print(f"Completed at: {datetime.now()}")
print(f"{'='*50}")